# Week 6 -- Function 6: GP + UCB Exploitation (5D)
**Cake Recipe Optimisation** — Goldilocks function, maximise Y toward zero. Extremes are punished.

In [1]:
# -- WEEKLY OBSERVATIONS LOG
import numpy as np

INITIAL_SIZE = 20
DATA_PATH_X  = '../data/function_6/initial_inputs.npy'
DATA_PATH_Y  = '../data/function_6/initial_outputs.npy'

# Goldilocks function: extremes are punished, sweet spot is in the interior
# W3 at [0.607, 0.167, 0.797, 0.720, 0.086] = -0.557 is the best
# W4 pushed to boundary extremes -> -0.982 (WORST ever)
weekly_log = [
    ([0.705342, 0.105077, 0.763664, 0.783759, 0.051342], -0.6958464033302365),  # W1
    ([0.770275, 0.188487, 0.822811, 0.738510, 0.068343], -0.7553482963799035),  # W2
    ([0.606716, 0.166787, 0.797421, 0.720111, 0.086136], -0.557288),            # W3 BEST
    ([0.406716, 0.005417, 0.997421, 0.920111, 0.000000], -0.981732150456315),   # W4 WORST (extremes)
    ([0.658400, 0.163211, 0.895070, 0.579545, 0.178551], -0.9140037485428404),  # W5
]

X_disk = np.load(DATA_PATH_X)
Y_disk = np.load(DATA_PATH_Y)

n_missing = (INITIAL_SIZE + len(weekly_log)) - len(Y_disk)
if n_missing > 0:
    new_entries = weekly_log[len(weekly_log) - n_missing:]
    for x_vals, y_val in new_entries:
        X_disk = np.vstack([X_disk, np.array([x_vals])])
        Y_disk = np.append(Y_disk, y_val)
    np.save(DATA_PATH_X, X_disk)
    np.save(DATA_PATH_Y, Y_disk)
    print(f'Synced {n_missing} new observation(s).')
else:
    print('Already up-to-date.')

print(f'X: {X_disk.shape} | Y: {Y_disk.shape}')
current_week = len(Y_disk) - INITIAL_SIZE + 1
print(f'Week {current_week} | {len(Y_disk)} total observations ({INITIAL_SIZE} initial + {len(Y_disk)-INITIAL_SIZE} added)')

Already up-to-date.
X: (25, 5) | Y: (25,)
Week 6 | 25 total observations (20 initial + 5 added)


In [2]:
# Cell 3: Load data and inspect -- Function 6: Cake Recipe Optimisation (5D), Maximise

X = np.load(DATA_PATH_X)
Y = np.load(DATA_PATH_Y)
n_dims = X.shape[1]

print(f'Input  shape : {X.shape}')
print(f'Output shape : {Y.shape}')
print()

# Sort by Y descending -- all negative, less negative = better
pairs = sorted(zip(Y.tolist(), X.tolist()), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 80)
print('  All observations (sorted descending by Y)')
print('=' * 80)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ('  <-- WORST' if i == len(Y_sorted)-1 else '')
    x_str = ', '.join(f'{v:.4f}' for v in x_val)
    print(f'  [{i+1:2d}]  X = [{x_str}]   Y = {y_val:+.4f}{marker}')
print('=' * 80)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
best_x_str = ', '.join(f'{v:.6f}' for v in best_X)
print(f'\n  Best Y*   = {best_Y:.6f}  (W3 sweet spot)')
print(f'  Best X*   = [{best_x_str}]')
print(f'  Worst Y   = {Y_sorted[-1]:.6f}  (W4 boundary extreme)')

Input  shape : (25, 5)
Output shape : (25,)

  All observations (sorted descending by Y)
  [ 1]  X = [0.6067, 0.1668, 0.7974, 0.7201, 0.0861]   Y = -0.5573  <-- best
  [ 2]  X = [0.7053, 0.1051, 0.7637, 0.7838, 0.0513]   Y = -0.6958
  [ 3]  X = [0.7282, 0.1547, 0.7326, 0.6940, 0.0564]   Y = -0.7143
  [ 4]  X = [0.7703, 0.1885, 0.8228, 0.7385, 0.0683]   Y = -0.7553
  [ 5]  X = [0.6188, 0.3318, 0.1873, 0.7562, 0.3288]   Y = -0.8292
  [ 6]  X = [0.6584, 0.1632, 0.8951, 0.5795, 0.1786]   Y = -0.9140
  [ 7]  X = [0.7829, 0.5363, 0.4433, 0.8597, 0.0103]   Y = -0.9358
  [ 8]  X = [0.4067, 0.0054, 0.9974, 0.9201, 0.0000]   Y = -0.9817
  [ 9]  X = [0.5368, 0.3088, 0.4119, 0.3882, 0.5225]   Y = -1.1448
  [10]  X = [0.2424, 0.8441, 0.5778, 0.6790, 0.5020]   Y = -1.2100
  [11]  X = [0.1451, 0.8967, 0.8963, 0.7263, 0.2363]   Y = -1.2338
  [12]  X = [0.7850, 0.9107, 0.7081, 0.9592, 0.0049]   Y = -1.2470
  [13]  X = [0.4322, 0.7156, 0.3418, 0.7050, 0.6150]   Y = -1.2942
  [14]  X = [0.7576, 0.3558, 0

In [3]:
# Cell 4: Fit GP
# F6 values in tight range -0.5 to -1.0 -- raw Y fit, no log, no scaler.
# length_scale=0.3: in 5D with sparse data, 0.1 leaves almost no connectivity.
# alpha=1e-4 for slight noise tolerance.
import warnings
warnings.filterwarnings('ignore')
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

kernel = RBF(length_scale=0.3, length_scale_bounds='fixed')
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-4)
gp.fit(X, Y)

w3_anchor = np.array([0.606716, 0.166787, 0.797421, 0.720111, 0.086136])

print('=' * 62)
print('  GP Fitting Results')
print('=' * 62)
print(f'  Kernel                  : {gp.kernel_}')
print(f'  Alpha (noise)           : 1e-4')
print(f'  Log-marginal-likelihood : {gp.log_marginal_likelihood_value_:.4f}')
pred_mean, pred_std = gp.predict(best_X.reshape(1, -1), return_std=True)
print('  Sanity check at best known X* (W3):')
print(f'    GP predicted mean      = {pred_mean[0]:.6f}')
print(f'    Actual Y* (W3)         = {best_Y:.6f}')
print(f'    GP predicted std       = {pred_std[0]:.8f}')
pred_w4, std_w4 = gp.predict(
    np.array([[0.406716, 0.005417, 0.997421, 0.920111, 0.000000]]), return_std=True)
print(f'  GP at W4 extreme [0.407, 0.005, 0.997, 0.920, 0.000]:')
print(f'    predicted mean         = {pred_w4[0]:.6f}  (actual: -0.9817, WORST)')
print('=' * 62)

  GP Fitting Results
  Kernel                  : RBF(length_scale=0.3)
  Alpha (noise)           : 1e-4
  Log-marginal-likelihood : -34.6253
  Sanity check at best known X* (W3):
    GP predicted mean      = -0.557624
    Actual Y* (W3)         = -0.557288
    GP predicted std       = 0.00999479
  GP at W4 extreme [0.407, 0.005, 0.997, 0.920, 0.000]:
    predicted mean         = -0.981603  (actual: -0.9817, WORST)


In [4]:
# Cell 5: GP UCB Acquisition -- focused random search around W3 anchor
# FIX: 12^5 structured grid caused corner artefact -- UCB picked all 5 dims at grid edge even at beta=0.3.
# Root cause: GP mean itself predicted best at the corner (-0.218 vs W3 actual -0.557).
# Random search has no systematic grid edges for UCB to latch onto.

w3_best = np.array([0.606716, 0.166787, 0.797421, 0.720111, 0.086136])
np.random.seed(42)
X_grid = np.clip(
    w3_best + np.random.uniform(-0.08, 0.08, size=(50000, n_dims)),
    0.0, 1.0
)

gp_mean, gp_std = gp.predict(X_grid, return_std=True)

beta = 1.0
ucb_scores = gp_mean + beta * gp_std

best_idx = np.argmax(ucb_scores)
next_x = X_grid[best_idx]
portal_string = '-'.join([f'{v:.6f}' for v in next_x])

print('=' * 72)
print('  GP UCB Acquisition (beta=1.0, focused random search)')
print('=' * 72)
w3_str = ', '.join(f'{v:.6f}' for v in w3_best)
print(f'  Anchor       : [{w3_str}]  (W3 best)')
print(f'  Search radius: ±0.08  |  50,000 random points  (no grid corners)')
print(f'  Beta         : {beta}  (exploitation)')
next_str = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Next query   : [{next_str}]')
print(f'  UCB score    : {ucb_scores[best_idx]:.6f}')
print(f'  GP mean      : {gp_mean[best_idx]:.6f}')
print(f'  GP std       : {gp_std[best_idx]:.6f}')
print('=' * 72)

  GP UCB Acquisition (beta=1.0, focused random search)
  Anchor       : [0.606716, 0.166787, 0.797421, 0.720111, 0.086136]  (W3 best)
  Search radius: ±0.08  |  50,000 random points  (no grid corners)
  Beta         : 1.0  (exploitation)
  Next query   : [0.527881, 0.234359, 0.727980, 0.790290, 0.010640]
  UCB score    : 0.111418
  GP mean      : -0.250132
  GP std       : 0.361550


In [5]:
# Cell 6: Sanity check -- distance to W3 anchor, no extremes, per-dimension comparison

w3_anchor    = np.array([0.606716, 0.166787, 0.797421, 0.720111, 0.086136])
dist_to_w3   = np.linalg.norm(next_x - w3_anchor)
dim_diffs    = np.abs(next_x - w3_anchor)

# Per-dimension deviation > 0.12 catches clipping artefacts (search is ±0.08, so >0.12 is unexpected)
dim_violations = [i + 1 for i, d in enumerate(dim_diffs) if d > 0.12]

# Extreme-value check: no dimension at or near 0/1 -- W4 extremes scored worst
extreme_dims = [i + 1 for i, v in enumerate(next_x) if v <= 0.001 or v >= 0.999]

print('=' * 72)
print('  Sanity Check (Goldilocks: interior, no extremes)')
print('=' * 72)
print(f'  Anchor   : [{" | ".join(f"{v:.6f}" for v in w3_anchor)}]  (W3, Y=-0.557)')
print(f'  Proposed : [{" | ".join(f"{v:.6f}" for v in next_x)}]')
print(f'  Distance : {dist_to_w3:.6f}')
print()
dim_names = ['x1', 'x2', 'x3', 'x4', 'x5']
for name, a_val, n_val, diff in zip(dim_names, w3_anchor, next_x, dim_diffs):
    ok = diff <= 0.12
    flag = '' if ok else '  <-- WARNING: > 0.12 from anchor'
    print(f'  {name}: anchor={a_val:.6f}  query={n_val:.6f}  diff={diff:.4f}{flag}')
print()
if dist_to_w3 > 0.20:
    print(f'  WARNING: distance {dist_to_w3:.4f} > 0.20 -- too far from W3 sweet spot.')
else:
    print(f'  OK: distance {dist_to_w3:.4f} within 0.20 of W3 anchor.')
if extreme_dims:
    print(f'  WARNING: dim(s) {extreme_dims} at 0.0 or 1.0 -- Goldilocks extremes are catastrophic!')
else:
    print(f'  OK: no dimensions at boundary extremes (all between 0.001 and 0.999).')
if dim_violations:
    print(f'  WARNING: dim(s) {dim_violations} deviate > 0.12 from anchor (clipping artefact?).')
else:
    print(f'  OK: all dimensions within 0.12 of anchor.')
print('=' * 72)

  Sanity Check (Goldilocks: interior, no extremes)
  Anchor   : [0.606716 | 0.166787 | 0.797421 | 0.720111 | 0.086136]  (W3, Y=-0.557)
  Proposed : [0.527881 | 0.234359 | 0.727980 | 0.790290 | 0.010640]
  Distance : 0.161950

  x1: anchor=0.606716  query=0.527881  diff=0.0788
  x2: anchor=0.166787  query=0.234359  diff=0.0676
  x3: anchor=0.797421  query=0.727980  diff=0.0694
  x4: anchor=0.720111  query=0.790290  diff=0.0702
  x5: anchor=0.086136  query=0.010640  diff=0.0755

  OK: distance 0.1619 within 0.20 of W3 anchor.
  OK: no dimensions at boundary extremes (all between 0.001 and 0.999).
  OK: all dimensions within 0.12 of anchor.


In [6]:
print('=' * 72)
print('  SUMMARY -- Week 6 Bayesian Optimisation')
print('=' * 72)
print('  Function        : 6 -- Cake Recipe Optimisation (5D)')
print('  Objective       : Maximise Y (all negative, want least negative)')
print(f'  Method          : GP UCB (beta=1.0, 5D grid, raw Y fit, length_scale=0.3)')
print(f'  Key insight     : Goldilocks -- interior sweet spot near W3, extremes punished')
print()
best_x_s = ', '.join(f'{v:.6f}' for v in best_X)
next_s   = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Best X* (W3)    : [{best_x_s}]')
print(f'  Best Y*         : {best_Y:.6f}')
print()
print(f'  Next query      : [{next_s}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 72)

  SUMMARY -- Week 6 Bayesian Optimisation
  Function        : 6 -- Cake Recipe Optimisation (5D)
  Objective       : Maximise Y (all negative, want least negative)
  Method          : GP UCB (beta=1.0, 5D grid, raw Y fit, length_scale=0.3)
  Key insight     : Goldilocks -- interior sweet spot near W3, extremes punished

  Best X* (W3)    : [0.606716, 0.166787, 0.797421, 0.720111, 0.086136]
  Best Y*         : -0.557288

  Next query      : [0.527881, 0.234359, 0.727980, 0.790290, 0.010640]

  Portal submission string:
  >>> 0.527881-0.234359-0.727980-0.790290-0.010640 <<<


## Week 6 -- Reflection

**Function**: F6 -- Cake Recipe Optimisation (5D)

### Strategy change from Week 5
- Removed NN: sent W4 to boundary extremes → -0.982 (worst ever).
- Removed SVM: 25 points in 5D, values in tight range — SVM can't meaningfully separate.
- Raw Y fit, no scaler, length_scale=0.3: correct ordering, GP connects sparse data.
- Grid stays in interior: all bounds away from 0.0 and 1.0 — Goldilocks rule.

### Pattern
| Week | Y | Notes |
|------|---|-------|
| W3 | -0.557 | Interior sweet spot — **best** |
| W1 | -0.696 | Slightly off from W3 |
| W2 | -0.755 | Higher x1/x3 than W1 — worse |
| W5 | -0.914 | Drifted from W3 region |
| W4 | -0.982 | Boundary extremes — **worst** |

### Query
- **Input submitted**: *(fill in portal submission string)*
- **Output received**: *(fill in after result)*
- **Best so far**: *(update after result)*

### What this result taught us
*(fill in after receiving result)*

### Strategy for Week 7
*(fill in after reflecting on result)*